# 05. KuaiRand Singleton No-Click Random-Exposure Model Comparison

This notebook applies the two KuaiRec-trained transfer models to KuaiRand random interactions and compares observed engagement metrics for rows selected by each model's score.

Models compared:

- **EU model**: predicts posterior expected utility `EU` and ranks by predicted EU.
- **Naive head score**: predicts four engagement heads and ranks by `p_complete + p_long + p_rewatch - p_neg`.
- **Completion-only head ranker**: uses the same head model but ranks only by `p_complete`.

The validation sample is restricted to rows that match the target sequential/autoplay UI as closely as KuaiRand allows:

1. Keep singleton random rows whose original exposure timestamp contains exactly one video, using `is_singleton_user_time == 1`.
2. Drop rows with `is_click == 1`, because clicked interactions are not behaviorally comparable to the target one-video autoplay UI.

In the current scored KuaiRand random sample, the singleton filter keeps **34,608 of 1,149,139 random rows**. Among those singleton rows, **8,459** have `is_click == 1` and are dropped, leaving **26,149** analysis rows, or **2.28%** of all random rows and **75.56%** of singleton random rows.

The notebook reports selected engagement metrics only: `watch_ratio`, `play_time_ms`, `is_comment`, `is_follow`, `is_forward`, `is_hate`, `is_like`, `is_profile_enter`, and `long_view`. Top-score recommendation metrics are shown for the top **5%**, **10%**, and **20%** rows within the cleaned singleton/no-click sample.

For uncertainty, the notebook adds a nonparametric row bootstrap with **500 draws**. For each ranker/top-percent/metric combination, the selected top-k rows are held fixed and the observed valid random rows in that selected set are resampled with replacement to estimate a 95% interval for the mean metric.

## Outputs

The notebook writes the final singleton/no-click random-row analysis only:

- `kuairand_random_head_naive_and_eu_scores.parquet`: full random-row inference cache with predicted head probabilities, naive head score, and predicted EU.
- `kuairand_singleton_noclick_random_head_naive_and_eu_scores.parquet`: optional singleton/no-click scored-row cache. It is skipped by default to avoid disk-space failures; the notebook computes this table in memory.
- `kuairand_singleton_noclick_random_selected_engagement_summary.csv`: long-format mean table for the selected engagement metrics.
- `kuairand_singleton_noclick_random_selected_engagement_comparison.csv`: model-vs-model comparison for each selected engagement metric and top quantile.
- `kuairand_singleton_noclick_random_selected_engagement_summary_wide.csv`: compact wide table for quick inspection.
- `kuairand_singleton_noclick_random_naive_reporting_table_top10_20.csv`: final no-CI reporting table for top 10/20 percent and the six selected metrics.
- `kuairand_singleton_noclick_random_naive_reporting_table_top10.csv`, `...top20.csv`: separate final no-CI reporting tables by top-percent level.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_draws.csv`: optional raw bootstrap draws. It is skipped by default because the CI table and plots are the reporting outputs.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_intervals.csv`: raw bootstrap interval table with score/rank metadata.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_table.csv`: clean reporting table of observed means, bootstrap SEs, and confidence intervals for all metrics, all 3 rankers, and top 5/10/20 percent.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_wide_by_top_pct.csv`: stacked paper-style wide tables with one row per metric and one column per recommender.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_wide_top5.csv`, `...top10.csv`, `...top20.csv`: separate paper-style tables for each top-percent level.
- `kuairand_singleton_noclick_random_bootstrap_plots/`: bootstrap distribution plots, one image per metric.
- `kuairand_singleton_noclick_random_selected_engagement_bootstrap_plots.pdf`: all bootstrap plots in one PDF.
- `kuairand_singleton_noclick_random_selected_engagement_diagnostics.json`: run diagnostics and filter counts.

The notebook also removes stale output files from earlier all-random / singleton-with-click versions if they are present, so the stored result folder reflects the current final specification.


In [ ]:
from pathlib import Path
import json
import math
import random
import shutil

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display

pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)

PROJECT_ROOT = Path("/Users/haozhangao/Desktop/RecSys Research")
KUAIRAND_DIR = PROJECT_ROOT / "python_code_KuaiRand"
MODEL_DIR = KUAIRAND_DIR / "model_outputs"
FEATURE_DIR = KUAIRAND_DIR / "outputs" / "feature_build"

INPUT_PATH = FEATURE_DIR / "kuairand_random_prediction_ranking_input.parquet"
HEAD_MODEL_PATH = MODEL_DIR / "kuairec_transfer_head_mlp_kuairand_features.pt"
EU_MODEL_PATH = MODEL_DIR / "kuairec_transfer_eu_mlp_kuairand_features.pt"

SCORED_OUTPUT_PATH = MODEL_DIR / "kuairand_random_head_naive_and_eu_scores.parquet"
SCORED_SINGLETON_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_head_naive_and_eu_scores.parquet"
SUMMARY_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_summary.csv"
COMPARISON_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_comparison.csv"
SUMMARY_WIDE_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_summary_wide.csv"
REPORTING_TABLE_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_naive_reporting_table_top10_20.csv"
REPORTING_TABLE_TOP_PATHS = {
    pct: MODEL_DIR / f"kuairand_singleton_noclick_random_naive_reporting_table_top{pct}.csv"
    for pct in [10, 20]
}
DIAGNOSTIC_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_diagnostics.json"
BOOTSTRAP_DRAWS_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_bootstrap_draws.csv"
BOOTSTRAP_INTERVALS_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_bootstrap_intervals.csv"
BOOTSTRAP_CI_TABLE_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_table.csv"
BOOTSTRAP_CI_WIDE_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_wide_by_top_pct.csv"
BOOTSTRAP_PLOT_DIR = MODEL_DIR / "kuairand_singleton_noclick_random_bootstrap_plots"
BOOTSTRAP_PDF_OUTPUT_PATH = MODEL_DIR / "kuairand_singleton_noclick_random_selected_engagement_bootstrap_plots.pdf"

SINGLETON_FILTER_COL = "is_singleton_user_time"
CLICK_FILTER_COL = "is_click"
RUN_INFERENCE = not SCORED_OUTPUT_PATH.exists()
OVERWRITE_SCORES = False
TOP_PCTS = [5, 10, 20]
BOOTSTRAP_CI_WIDE_TOP_PATHS = {
    pct: MODEL_DIR / f"kuairand_singleton_noclick_random_selected_engagement_bootstrap_ci_wide_top{pct}.csv"
    for pct in TOP_PCTS
}
RANDOM_SEED = 20260706
BOOTSTRAP_N = 500
BOOTSTRAP_ALPHA = 0.05
BOOTSTRAP_RANDOM_SEED = RANDOM_SEED + 500
CHUNK_ROWS = 200_000
PREDICT_BATCH_SIZE = 131_072

# Disk-light defaults: the final report needs the summary/CI outputs, not the large row-level optional caches.
WRITE_FILTERED_SCORED_ROWS = False
WRITE_BOOTSTRAP_DRAWS = False

STALE_RESULT_PATHS = [
    MODEL_DIR / "kuairand_all_random_selected_engagement_summary.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_comparison.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_summary_wide.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_bootstrap_draws.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_bootstrap_intervals.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_bootstrap_ci_table.csv",
    MODEL_DIR / "kuairand_all_random_selected_engagement_bootstrap_plots.pdf",
    MODEL_DIR / "kuairand_all_random_selected_engagement_diagnostics.json",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_summary.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_comparison.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_summary_wide.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_bootstrap_draws.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_bootstrap_intervals.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_bootstrap_ci_table.csv",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_bootstrap_plots.pdf",
    MODEL_DIR / "kuairand_singleton_random_selected_engagement_diagnostics.json",
]
STALE_RESULT_DIRS = [
    MODEL_DIR / "kuairand_all_random_bootstrap_plots",
    MODEL_DIR / "kuairand_singleton_random_bootstrap_plots",
]

removed_stale_outputs = []
for stale_path in STALE_RESULT_PATHS:
    if stale_path.exists():
        stale_path.unlink()
        removed_stale_outputs.append(str(stale_path))
for stale_dir in STALE_RESULT_DIRS:
    if stale_dir.exists():
        shutil.rmtree(stale_dir)
        removed_stale_outputs.append(str(stale_dir))

OPTIONAL_OUTPUT_PATHS = {
    "filtered_scored_rows": SCORED_SINGLETON_OUTPUT_PATH,
    "bootstrap_draws": BOOTSTRAP_DRAWS_OUTPUT_PATH,
}
removed_optional_outputs = []
if not WRITE_FILTERED_SCORED_ROWS and SCORED_SINGLETON_OUTPUT_PATH.exists():
    SCORED_SINGLETON_OUTPUT_PATH.unlink()
    removed_optional_outputs.append(str(SCORED_SINGLETON_OUTPUT_PATH))
if not WRITE_BOOTSTRAP_DRAWS and BOOTSTRAP_DRAWS_OUTPUT_PATH.exists():
    BOOTSTRAP_DRAWS_OUTPUT_PATH.unlink()
    removed_optional_outputs.append(str(BOOTSTRAP_DRAWS_OUTPUT_PATH))


DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

required = [INPUT_PATH, HEAD_MODEL_PATH, EU_MODEL_PATH]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("device:", DEVICE)
print("input:", INPUT_PATH)
print("head model:", HEAD_MODEL_PATH)
print("EU model:", EU_MODEL_PATH)
print("run inference:", RUN_INFERENCE)
print("top percentages:", TOP_PCTS)
print("bootstrap draws:", BOOTSTRAP_N)
print("write filtered scored rows:", WRITE_FILTERED_SCORED_ROWS)
print("write raw bootstrap draws:", WRITE_BOOTSTRAP_DRAWS)
print("bootstrap CI table:", BOOTSTRAP_CI_TABLE_OUTPUT_PATH)
print("bootstrap CI wide table:", BOOTSTRAP_CI_WIDE_OUTPUT_PATH)
print("singleton filter:", SINGLETON_FILTER_COL)
print("click exclusion filter:", f"{CLICK_FILTER_COL} != 1")

In [ ]:
class EdgeFeatureEncoder(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, embedding_dims):
        super().__init__()
        self.n_cont = int(n_cont)
        self.n_bin = int(n_bin)
        self.categorical_cardinalities = [int(c) for c in categorical_cardinalities]
        self.embedding_dims = [int(d) for d in embedding_dims]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, dim)
            for cardinality, dim in zip(self.categorical_cardinalities, self.embedding_dims)
        ])
        self.output_dim = self.n_cont + self.n_bin + sum(self.embedding_dims)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        parts = []
        if self.n_cont:
            parts.append(x_cont_batch.float())
        if self.n_bin:
            parts.append(x_bin_batch.float())
        for j, emb in enumerate(self.embeddings):
            codes = x_cat_batch[:, j].long().clamp(min=0, max=emb.num_embeddings - 1)
            parts.append(emb(codes))
        return torch.cat(parts, dim=1)


class TransferMLP(nn.Module):
    def __init__(self, n_cont, n_bin, categorical_cardinalities, embedding_dims, hidden_sizes, dropout, out_dim):
        super().__init__()
        self.encoder = EdgeFeatureEncoder(n_cont, n_bin, categorical_cardinalities, embedding_dims)
        layers = []
        dim = self.encoder.output_dim
        for hidden in hidden_sizes:
            layers.append(nn.Linear(dim, int(hidden)))
            layers.append(nn.SiLU())
            layers.append(nn.Dropout(float(dropout)))
            dim = int(hidden)
        layers.append(nn.Linear(dim, int(out_dim)))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cont_batch, x_bin_batch, x_cat_batch):
        return self.mlp(self.encoder(x_cont_batch, x_bin_batch, x_cat_batch))


def load_transfer_model(path):
    payload = torch.load(path, map_location="cpu", weights_only=False)
    spec = payload["feature_spec"]
    arch = spec["architecture"]
    meta = spec["selected_feature_metadata"]
    model = TransferMLP(
        n_cont=len(spec["continuous_cols"]),
        n_bin=len(spec["binary_cols"]),
        categorical_cardinalities=meta["categorical_cardinalities"],
        embedding_dims=arch["embedding_dims"],
        hidden_sizes=arch["hidden_sizes"],
        dropout=arch["dropout"],
        out_dim=arch["output_dim"],
    )
    model.load_state_dict(payload["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return payload, spec, model

head_payload, head_spec, head_model = load_transfer_model(HEAD_MODEL_PATH)
eu_payload, eu_spec, eu_model = load_transfer_model(EU_MODEL_PATH)

if head_spec["feature_cols"] != eu_spec["feature_cols"]:
    raise ValueError("Head and EU models do not use the same feature columns.")

feature_cols = list(head_spec["feature_cols"])
continuous_cols = list(head_spec["continuous_cols"])
binary_cols = list(head_spec["binary_cols"])
categorical_cols = list(head_spec["categorical_cols"])
selected_meta = head_spec["selected_feature_metadata"]
head_labels = list(head_spec["label_cols"])
expected_head_labels = ["y_complete", "y_long", "y_rewatch", "y_neg"]
if head_labels != expected_head_labels:
    raise ValueError(f"Unexpected head label order: {head_labels}")

eu_target_mean = float(eu_spec["target_standardization"]["mean"])
eu_target_std = float(eu_spec["target_standardization"]["std"])

print("features:", len(feature_cols))
print("continuous/binary/categorical:", len(continuous_cols), len(binary_cols), len(categorical_cols))
print("head labels:", head_labels)
print("EU target mean/std:", eu_target_mean, eu_target_std)

In [ ]:
def stable_hash_codes(values, n_buckets):
    hashed = pd.util.hash_pandas_object(values.astype("string").fillna("__MISSING__"), index=False)
    return (hashed.to_numpy(dtype="uint64") % np.uint64(n_buckets)).astype("int64") + 1


def string_candidates_for_mapping(series):
    base = series.astype("string").fillna("__MISSING__").replace({"": "__MISSING__"})
    candidates = [base]
    numeric = pd.to_numeric(series, errors="coerce")
    finite = numeric.notna()
    if finite.any():
        numeric_float = numeric.astype(float)
        int_like = finite & np.isclose(numeric_float, np.round(numeric_float))

        as_int = pd.Series("__MISSING__", index=series.index, dtype="string")
        as_int.loc[int_like] = np.round(numeric.loc[int_like].astype(float)).astype("int64").astype("string")
        as_int.loc[finite & ~int_like] = numeric.loc[finite & ~int_like].astype("string")
        candidates.append(as_int)

        as_float1 = pd.Series("__MISSING__", index=series.index, dtype="string")
        as_float1.loc[int_like] = np.round(numeric.loc[int_like].astype(float)).astype("int64").astype("string") + ".0"
        as_float1.loc[finite & ~int_like] = numeric.loc[finite & ~int_like].astype("string")
        candidates.append(as_float1)
    return candidates


def transform_continuous(frame):
    out = np.empty((len(frame), len(continuous_cols)), dtype=np.float32)
    scaler = selected_meta["continuous_scaler"]
    missing_counts = {}
    for j, col in enumerate(continuous_cols):
        raw = pd.to_numeric(frame[col], errors="coerce").astype("float32")
        mean = float(scaler[col]["mean"])
        std = float(scaler[col]["std"])
        if not np.isfinite(std) or std <= 1e-8:
            std = 1.0
        missing_counts[col] = int(raw.isna().sum())
        out[:, j] = ((raw.fillna(mean) - mean) / (std + 1e-8)).to_numpy(dtype=np.float32)
    return torch.from_numpy(out), missing_counts


def transform_binary(frame):
    out = np.empty((len(frame), len(binary_cols)), dtype=np.float32)
    missing_counts = {}
    for j, col in enumerate(binary_cols):
        raw = pd.to_numeric(frame[col], errors="coerce")
        missing_counts[col] = int(raw.isna().sum())
        out[:, j] = raw.fillna(0).clip(0, 1).to_numpy(dtype=np.float32)
    return torch.from_numpy(out), missing_counts


def transform_categorical(frame):
    out = np.empty((len(frame), len(categorical_cols)), dtype=np.int64)
    unseen_counts = {}
    missing_counts = {}
    mappings = selected_meta["categorical_mappings"]
    hash_bucket_size = int(selected_meta.get("hash_bucket_size") or 10000)

    for j, col in enumerate(categorical_cols):
        info = mappings[col]
        missing_counts[col] = int(frame[col].isna().sum())
        if info.get("encoding") == "hash_embedding" or info.get("mapping") is None:
            values = frame[col].astype("string").fillna("__MISSING__").replace({"": "__MISSING__"})
            codes = stable_hash_codes(values, hash_bucket_size)
        else:
            mapping = info["mapping"]
            code_series = None
            for values in string_candidates_for_mapping(frame[col]):
                mapped = values.map(mapping)
                code_series = mapped if code_series is None else code_series.fillna(mapped)
            codes = code_series.fillna(0).astype("int64").to_numpy()
        cardinality = int(info["cardinality"])
        codes = np.clip(codes, 0, cardinality - 1).astype(np.int64)
        unseen_counts[col] = int((codes == 0).sum())
        out[:, j] = codes
    return torch.from_numpy(out).long(), unseen_counts, missing_counts


def transform_feature_frame(frame):
    missing = sorted(set(feature_cols) - set(frame.columns))
    if missing:
        raise KeyError(f"Scoring frame is missing required feature columns: {missing}")
    x_cont, cont_missing = transform_continuous(frame)
    x_bin, bin_missing = transform_binary(frame)
    x_cat, cat_unseen, cat_missing = transform_categorical(frame)
    audit = {
        "continuous_missing": cont_missing,
        "binary_missing": bin_missing,
        "categorical_missing": cat_missing,
        "categorical_unseen_or_unknown": cat_unseen,
    }
    return x_cont, x_bin, x_cat, audit


def predict_models(x_cont, x_bin, x_cat):
    n = int(x_cont.shape[0])
    head_chunks = []
    eu_chunks = []
    with torch.no_grad():
        for start in range(0, n, PREDICT_BATCH_SIZE):
            stop = min(start + PREDICT_BATCH_SIZE, n)
            xb_cont = x_cont[start:stop].to(DEVICE)
            xb_bin = x_bin[start:stop].to(DEVICE)
            xb_cat = x_cat[start:stop].to(DEVICE)

            head_logits = head_model(xb_cont, xb_bin, xb_cat)
            head_prob = torch.sigmoid(head_logits).detach().cpu().numpy().astype(np.float32)

            eu_std = eu_model(xb_cont, xb_bin, xb_cat).detach().cpu().numpy().reshape(-1).astype(np.float32)
            eu_raw = eu_std * np.float32(eu_target_std) + np.float32(eu_target_mean)

            head_chunks.append(head_prob)
            eu_chunks.append(eu_raw.astype(np.float32))
    return np.vstack(head_chunks), np.concatenate(eu_chunks)

print("encoding and prediction helpers ready")

In [ ]:
def score_random_rows():
    schema_cols = pq.ParquetFile(INPUT_PATH).schema_arrow.names
    id_cols = [
        "random_row_id", "rank_group_user_id", "rank_group_session_id", "rank_group_user_timestamp_id",
        "user_id", "video_id", "date", "hourmin", "time_ms", "time", "timestamp",
        "session_id", "session_index", "tab", "event_block_size", "event_block_size_by_tab",
        "is_singleton_user_time", "is_singleton_user_time_tab", "position_in_session_or_event_order",
    ]
    outcome_cols = [
        "is_rand", "source", "log_source", "play_time_ms", "duration_ms", "duration_for_ratio_ms",
        "watch_ratio", "is_click", "is_like", "is_follow", "is_comment", "is_forward", "is_hate",
        "long_view", "profile_stay_time", "comment_stay_time", "is_profile_enter",
        "y_complete", "y_long", "y_rewatch", "y_neg",
    ]
    keep_cols = [col for col in id_cols + outcome_cols if col in schema_cols]
    read_cols = list(dict.fromkeys(keep_cols + feature_cols))

    pf = pq.ParquetFile(INPUT_PATH)
    writer = None
    rows_written = 0
    audit_records = []

    for chunk_idx, batch in enumerate(pf.iter_batches(batch_size=CHUNK_ROWS, columns=read_cols), start=1):
        frame = batch.to_pandas()
        x_cont, x_bin, x_cat, audit = transform_feature_frame(frame)
        head_prob, eu_pred = predict_models(x_cont, x_bin, x_cat)

        out = frame[keep_cols].copy()
        for j, label in enumerate(head_labels):
            suffix = label.replace("y_", "")
            out[f"pred_p_{suffix}"] = head_prob[:, j].astype(np.float32)
        out["score_head_naive"] = (
            out["pred_p_complete"] + out["pred_p_long"] + out["pred_p_rewatch"] - out["pred_p_neg"]
        ).astype(np.float32)
        out["pred_EU"] = eu_pred.astype(np.float32)

        table = pa.Table.from_pandas(out, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(SCORED_OUTPUT_PATH, table.schema)
        writer.write_table(table)
        rows_written += len(out)

        for section, values in audit.items():
            for col, count in values.items():
                audit_records.append({
                    "chunk": chunk_idx,
                    "section": section,
                    "feature": col,
                    "count": int(count),
                    "rows": int(len(out)),
                })
        print(f"chunk {chunk_idx}: scored {len(out):,} rows; cumulative {rows_written:,}")

    if writer is not None:
        writer.close()
    else:
        raise RuntimeError("No rows were written")

    return pd.DataFrame(audit_records)

if RUN_INFERENCE:
    if SCORED_OUTPUT_PATH.exists() and not OVERWRITE_SCORES:
        print("Using existing scored output:", SCORED_OUTPUT_PATH)
        feature_audit = pd.DataFrame()
    else:
        feature_audit = score_random_rows()
        print("saved scored rows:", SCORED_OUTPUT_PATH)
else:
    if not SCORED_OUTPUT_PATH.exists():
        raise FileNotFoundError(f"RUN_INFERENCE=False but scored output does not exist: {SCORED_OUTPUT_PATH}")
    feature_audit = pd.DataFrame()
    print("Skipping inference and using:", SCORED_OUTPUT_PATH)

In [ ]:
raw_pred_df = pd.read_parquet(SCORED_OUTPUT_PATH)
raw_n = len(raw_pred_df)

for required_col in [SINGLETON_FILTER_COL, CLICK_FILTER_COL]:
    if required_col not in raw_pred_df.columns:
        raise KeyError(f"Scored rows are missing required filter column: {required_col}")

singleton_flag = pd.to_numeric(raw_pred_df[SINGLETON_FILTER_COL], errors="coerce").fillna(0).astype("int8").eq(1)
click_flag = pd.to_numeric(raw_pred_df[CLICK_FILTER_COL], errors="coerce").fillna(0).astype("int8").eq(1)
analysis_flag = singleton_flag & ~click_flag

singleton_n = int(singleton_flag.sum())
singleton_click_n = int((singleton_flag & click_flag).sum())
analysis_n = int(analysis_flag.sum())
if singleton_n == 0:
    raise ValueError(f"No rows satisfy {SINGLETON_FILTER_COL} == 1")
if analysis_n == 0:
    raise ValueError(f"No rows satisfy {SINGLETON_FILTER_COL} == 1 and {CLICK_FILTER_COL} != 1")

pred_df = raw_pred_df.loc[analysis_flag].copy()
del raw_pred_df
n = len(pred_df)

MODEL_SCORE_SPECS = [
    ("predicted_EU", "pred_EU", "rank_EU_singleton_noclick"),
    ("head_naive_score", "score_head_naive", "rank_head_naive_singleton_noclick"),
    ("head_completion_only", "pred_p_complete", "rank_head_completion_only_singleton_noclick"),
]

for method, score_col, rank_col in MODEL_SCORE_SPECS:
    pct_col = f"pct_{rank_col}"
    rank = pred_df[score_col].rank(method="first", ascending=False).astype("int32")
    pred_df[rank_col] = rank
    pred_df[pct_col] = (rank.astype("float64") / float(n)).astype("float32")

if "rank_group_user_timestamp_id" in pred_df.columns:
    rank_group_cols = ["rank_group_user_timestamp_id"]
elif {"user_id", "time_ms"}.issubset(pred_df.columns):
    rank_group_cols = ["user_id", "time_ms"]
else:
    rank_group_cols = []

if rank_group_cols:
    pred_df["rank_group_size"] = pred_df.groupby(rank_group_cols, sort=False)["pred_EU"].transform("size").astype("int32")
    pred_df["rank_head_naive_within_group"] = pred_df.groupby(rank_group_cols, sort=False)["score_head_naive"].rank(method="first", ascending=False).astype("int32")
    pred_df["rank_head_completion_only_within_group"] = pred_df.groupby(rank_group_cols, sort=False)["pred_p_complete"].rank(method="first", ascending=False).astype("int32")
    pred_df["rank_EU_within_group"] = pred_df.groupby(rank_group_cols, sort=False)["pred_EU"].rank(method="first", ascending=False).astype("int32")
else:
    pred_df["rank_group_size"] = 1

if WRITE_FILTERED_SCORED_ROWS:
    pred_df.to_parquet(SCORED_SINGLETON_OUTPUT_PATH, index=False)
    saved_singleton_scored_rows_path = str(SCORED_SINGLETON_OUTPUT_PATH)
else:
    saved_singleton_scored_rows_path = None
    print("skipped saving filtered scored rows to avoid disk-space pressure:", SCORED_SINGLETON_OUTPUT_PATH)

cohort_summary = pd.DataFrame([{
    "cohort": "singleton_noclick_random",
    "filter": f"{SINGLETON_FILTER_COL} == 1 and {CLICK_FILTER_COL} != 1",
    "full_random_rows": int(raw_n),
    "singleton_rows_before_click_filter": int(singleton_n),
    "singleton_click_rows_dropped": int(singleton_click_n),
    "analysis_rows": int(n),
    "analysis_share_of_all_random": float(n / raw_n),
    "analysis_share_of_singleton_random": float(n / singleton_n),
    "users": int(pred_df["user_id"].nunique()) if "user_id" in pred_df.columns else None,
    "videos": int(pred_df["video_id"].nunique()) if "video_id" in pred_df.columns else None,
}])
display(cohort_summary)

if "event_block_size" in pred_df.columns:
    display(pred_df["event_block_size"].value_counts(dropna=False).sort_index().to_frame("analysis_rows"))

score_preview_cols = [
    "pred_EU", "score_head_naive", "pred_p_complete", "pred_p_long", "pred_p_rewatch", "pred_p_neg",
    "watch_ratio", "play_time_ms", "is_comment", "is_follow", "is_forward",
    "is_hate", "is_like", "is_profile_enter", "long_view",
]
display(pred_df[[c for c in score_preview_cols if c in pred_df.columns]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))

print("full random rows:", f"{raw_n:,}")
print("singleton random rows before click filter:", f"{singleton_n:,}", f"({singleton_n / raw_n:.2%} of all random)")
print("singleton rows dropped for click=1:", f"{singleton_click_n:,}", f"({singleton_click_n / singleton_n:.2%} of singleton)")
print("analysis rows after click filter:", f"{n:,}", f"({n / raw_n:.2%} of all random; {n / singleton_n:.2%} of singleton)")
if saved_singleton_scored_rows_path:
    print("saved analysis scored rows:", saved_singleton_scored_rows_path)
else:
    print("analysis scored rows kept in memory only")

In [ ]:
ENGAGEMENT_METRICS = [
    "watch_ratio",
    "play_time_ms",
    "is_comment",
    "is_follow",
    "is_forward",
    "is_hate",
    "is_like",
    "is_profile_enter",
    "long_view",
]
ENGAGEMENT_METRICS = [col for col in ENGAGEMENT_METRICS if col in pred_df.columns]

for col in ENGAGEMENT_METRICS:
    pred_df[col] = pd.to_numeric(pred_df[col], errors="coerce")

baseline = {
    "method": "all_singleton_noclick_random",
    "top_pct": 100,
    "n": int(n),
    "share": 1.0,
    "score_col": None,
    "score_threshold": np.nan,
    "score_mean": np.nan,
}
for metric in ENGAGEMENT_METRICS:
    baseline[f"mean_{metric}"] = float(pred_df[metric].mean())

summary_rows = [baseline]
for method, score_col, rank_col in MODEL_SCORE_SPECS:
    for pct in TOP_PCTS:
        k = int(math.ceil(n * pct / 100.0))
        sub = pred_df.loc[pred_df[rank_col] <= k]
        row = {
            "method": method,
            "top_pct": int(pct),
            "n": int(len(sub)),
            "share": float(len(sub) / n),
            "score_col": score_col,
            "score_threshold": float(sub[score_col].min()),
            "score_mean": float(sub[score_col].mean()),
        }
        for metric in ENGAGEMENT_METRICS:
            row[f"mean_{metric}"] = float(sub[metric].mean())
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False)

display(summary_df)
print("saved long summary:", SUMMARY_OUTPUT_PATH)

In [ ]:
comparison_rows = []
base = summary_df.loc[summary_df["method"].eq("all_singleton_noclick_random")].iloc[0]
method_names = [method for method, _, _ in MODEL_SCORE_SPECS]

for pct in TOP_PCTS:
    method_rows = {
        method: summary_df.loc[(summary_df["method"].eq(method)) & (summary_df["top_pct"].eq(pct))].iloc[0]
        for method in method_names
    }
    for metric in ENGAGEMENT_METRICS:
        col = f"mean_{metric}"
        row = {
            "metric": metric,
            "top_pct": int(pct),
            "all_singleton_noclick_random": float(base[col]),
        }
        for method, method_row in method_rows.items():
            row[method] = float(method_row[col])
            row[f"{method}_minus_all_singleton_noclick"] = float(method_row[col] - base[col])
            row[f"{method}_n"] = int(method_row["n"])
        row["EU_minus_naive"] = row["predicted_EU"] - row["head_naive_score"]
        row["EU_minus_completion"] = row["predicted_EU"] - row["head_completion_only"]
        row["naive_minus_completion"] = row["head_naive_score"] - row["head_completion_only"]
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_OUTPUT_PATH, index=False)

summary_wide = comparison_df[[
    "top_pct", "metric", "all_singleton_noclick_random",
    "predicted_EU", "head_naive_score", "head_completion_only",
    "EU_minus_naive", "EU_minus_completion", "naive_minus_completion",
    "predicted_EU_minus_all_singleton_noclick", "head_naive_score_minus_all_singleton_noclick", "head_completion_only_minus_all_singleton_noclick",
]].copy()
summary_wide.to_csv(SUMMARY_WIDE_OUTPUT_PATH, index=False)

print("saved full comparison:", COMPARISON_OUTPUT_PATH)
print("saved wide comparison:", SUMMARY_WIDE_OUTPUT_PATH)
display(summary_wide)

In [ ]:
REPORT_TOP_PCTS = [10, 20]
REPORT_METRICS = [
    "is_follow",
    "is_hate",
    "is_like",
    "is_profile_enter",
    "play_time_ms",
    "watch_ratio",
]
REPORT_METHOD_LABELS = {
    "predicted_EU": "EU",
    "head_naive_score": "Naive heads",
    "head_completion_only": "Completion only",
}

report_numeric = (
    comparison_df.loc[
        comparison_df["top_pct"].isin(REPORT_TOP_PCTS)
        & comparison_df["metric"].isin(REPORT_METRICS),
        ["top_pct", "metric", "predicted_EU", "head_naive_score", "head_completion_only"],
    ]
    .copy()
)
report_numeric["metric"] = pd.Categorical(report_numeric["metric"], categories=REPORT_METRICS, ordered=True)
report_numeric = report_numeric.sort_values(["top_pct", "metric"]).reset_index(drop=True)
report_numeric = report_numeric.rename(columns=REPORT_METHOD_LABELS)
report_numeric.to_csv(REPORTING_TABLE_OUTPUT_PATH, index=False)

report_display_tables = {}
for pct in REPORT_TOP_PCTS:
    table = report_numeric.loc[report_numeric["top_pct"].eq(pct), ["metric", "EU", "Naive heads", "Completion only"]].copy()
    table.to_csv(REPORTING_TABLE_TOP_PATHS[pct], index=False)
    report_display_tables[pct] = table

print("Final no-CI reporting tables: top 10/20 percent, six selected metrics, naive head weights.")
print("saved combined reporting table:", REPORTING_TABLE_OUTPUT_PATH)
for pct, table in report_display_tables.items():
    print(f"saved top {pct}% reporting table:", REPORTING_TABLE_TOP_PATHS[pct])
    print(f"Top {pct}% mean metrics")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 180):
        display(table)


## Bootstrap Intervals and Plots

For each ranker, top-percent level, and selected engagement metric, the notebook bootstraps the mean over the selected singleton/no-click random rows. This does not rerun model inference; it reuses the fixed predicted scores and fixed top-k selections from the scored cache.

In [ ]:
BOOTSTRAP_PLOT_DIR.mkdir(parents=True, exist_ok=True)

RANKER_LABELS = {
    "predicted_EU": "EU",
    "head_naive_score": "Naive heads",
    "head_completion_only": "Completion only",
}
RANKER_COLORS = {
    "predicted_EU": "#1b9e77",
    "head_naive_score": "#d95f02",
    "head_completion_only": "#7570b3",
}
RANKER_ORDER = [method for method, _, _ in MODEL_SCORE_SPECS]
METRIC_ORDER = list(ENGAGEMENT_METRICS)

rng = np.random.default_rng(BOOTSTRAP_RANDOM_SEED)
bootstrap_draw_rows = []
bootstrap_interval_rows = []


def bootstrap_mean_draws(values, n_draws, rng):
    arr = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy(dtype=np.float64)
    n_obs = int(arr.size)
    if n_obs == 0:
        return np.full(n_draws, np.nan, dtype=np.float64), n_obs
    sample_idx = rng.integers(0, n_obs, size=(n_draws, n_obs), endpoint=False)
    draws = arr[sample_idx].mean(axis=1)
    return draws, n_obs


for method, score_col, rank_col in MODEL_SCORE_SPECS:
    for pct in TOP_PCTS:
        k = int(math.ceil(n * pct / 100.0))
        selected = pred_df.loc[pred_df[rank_col] <= k]
        for metric in ENGAGEMENT_METRICS:
            values = selected[metric]
            draws, n_metric = bootstrap_mean_draws(values, BOOTSTRAP_N, rng)
            observed_mean = float(pd.to_numeric(values, errors="coerce").mean())
            ci_lower, ci_upper = np.nanpercentile(draws, [100 * BOOTSTRAP_ALPHA / 2, 100 * (1 - BOOTSTRAP_ALPHA / 2)])
            boot_mean = float(np.nanmean(draws))
            boot_se = float(np.nanstd(draws, ddof=1))

            bootstrap_interval_rows.append({
                "method": method,
                "ranker_label": RANKER_LABELS.get(method, method),
                "score_col": score_col,
                "rank_col": rank_col,
                "top_pct": int(pct),
                "selected_n": int(len(selected)),
                "metric": metric,
                "metric_n": n_metric,
                "observed_mean": observed_mean,
                "bootstrap_mean": boot_mean,
                "bootstrap_se": boot_se,
                "ci_lower": float(ci_lower),
                "ci_upper": float(ci_upper),
                "bootstrap_n": int(BOOTSTRAP_N),
                "alpha": float(BOOTSTRAP_ALPHA),
            })
            for draw_idx, draw_value in enumerate(draws, start=1):
                bootstrap_draw_rows.append({
                    "method": method,
                    "ranker_label": RANKER_LABELS.get(method, method),
                    "top_pct": int(pct),
                    "metric": metric,
                    "bootstrap_draw": int(draw_idx),
                    "mean_value": float(draw_value),
                })

bootstrap_draws_df = pd.DataFrame(bootstrap_draw_rows)
bootstrap_intervals_df = pd.DataFrame(bootstrap_interval_rows)
bootstrap_intervals_df["method_order"] = bootstrap_intervals_df["method"].map({m: i for i, m in enumerate(RANKER_ORDER)})
bootstrap_intervals_df["metric_order"] = bootstrap_intervals_df["metric"].map({m: i for i, m in enumerate(METRIC_ORDER)})
bootstrap_intervals_df = bootstrap_intervals_df.sort_values(["metric_order", "top_pct", "method_order"]).drop(columns=["metric_order", "method_order"])

bootstrap_ci_table_df = bootstrap_intervals_df[[
    "metric",
    "top_pct",
    "method",
    "ranker_label",
    "selected_n",
    "metric_n",
    "observed_mean",
    "bootstrap_mean",
    "bootstrap_se",
    "ci_lower",
    "ci_upper",
    "bootstrap_n",
    "alpha",
]].copy()
bootstrap_ci_table_df["ci_level"] = f"{100 * (1 - BOOTSTRAP_ALPHA):.0f}%"
bootstrap_ci_table_df["mean_ci"] = bootstrap_ci_table_df.apply(
    lambda row: f"{row['observed_mean']:.6g} [{row['ci_lower']:.6g}, {row['ci_upper']:.6g}]",
    axis=1,
)

if WRITE_BOOTSTRAP_DRAWS:
    bootstrap_draws_df.to_csv(BOOTSTRAP_DRAWS_OUTPUT_PATH, index=False)
else:
    print("skipped saving raw bootstrap draws to save disk space:", BOOTSTRAP_DRAWS_OUTPUT_PATH)
bootstrap_intervals_df.to_csv(BOOTSTRAP_INTERVALS_OUTPUT_PATH, index=False)
bootstrap_ci_table_df.to_csv(BOOTSTRAP_CI_TABLE_OUTPUT_PATH, index=False)

# Paper-style tables: one table for each top-percent level. Rows are metrics,
# columns are recommenders, and cells are mean [95% CI].
wide_ci_tables = {}
wide_ci_stack_rows = []
ranker_column_order = [RANKER_LABELS[m] for m in RANKER_ORDER]

for pct in TOP_PCTS:
    pct_table = (
        bootstrap_ci_table_df.loc[bootstrap_ci_table_df["top_pct"].eq(pct)]
        .pivot(index="metric", columns="ranker_label", values="mean_ci")
        .reindex(index=METRIC_ORDER, columns=ranker_column_order)
        .reset_index()
    )
    wide_ci_tables[pct] = pct_table
    pct_table.to_csv(BOOTSTRAP_CI_WIDE_TOP_PATHS[pct], index=False)

    pct_stack = pct_table.copy()
    pct_stack.insert(0, "top_pct", int(pct))
    wide_ci_stack_rows.append(pct_stack)

bootstrap_ci_wide_df = pd.concat(wide_ci_stack_rows, ignore_index=True)
bootstrap_ci_wide_df.to_csv(BOOTSTRAP_CI_WIDE_OUTPUT_PATH, index=False)

print("Bootstrap confidence intervals for all selected engagement metrics, 3 rankers, and top 5/10/20 percent.")
print("Rows:", len(bootstrap_ci_table_df))
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 240):
    display(bootstrap_ci_table_df)

if WRITE_BOOTSTRAP_DRAWS:
    print("saved bootstrap draws:", BOOTSTRAP_DRAWS_OUTPUT_PATH)
else:
    print("raw bootstrap draws were not saved; plots use the in-memory draws from this run")
print("saved raw bootstrap intervals:", BOOTSTRAP_INTERVALS_OUTPUT_PATH)
print("saved reporting bootstrap CI table:", BOOTSTRAP_CI_TABLE_OUTPUT_PATH)
print("saved combined wide bootstrap CI table:", BOOTSTRAP_CI_WIDE_OUTPUT_PATH)
for pct in TOP_PCTS:
    print(f"saved top {pct}% wide bootstrap CI table:", BOOTSTRAP_CI_WIDE_TOP_PATHS[pct])
    print(f"Top {pct}%: mean [95% CI]")
    with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", 240):
        display(wide_ci_tables[pct])


In [ ]:
metric_plot_paths = []

with PdfPages(BOOTSTRAP_PDF_OUTPUT_PATH) as pdf:
    for metric in ENGAGEMENT_METRICS:
        fig, axes = plt.subplots(1, len(TOP_PCTS), figsize=(5.2 * len(TOP_PCTS), 4.2), dpi=150, sharey=False)
        if len(TOP_PCTS) == 1:
            axes = [axes]
        metric_draws = bootstrap_draws_df.loc[bootstrap_draws_df["metric"].eq(metric)]
        metric_intervals = bootstrap_intervals_df.loc[bootstrap_intervals_df["metric"].eq(metric)]

        for ax, pct in zip(axes, TOP_PCTS):
            pct_draws = metric_draws.loc[metric_draws["top_pct"].eq(pct)]
            for method, _, _ in MODEL_SCORE_SPECS:
                sub = pct_draws.loc[pct_draws["method"].eq(method), "mean_value"].dropna().to_numpy(dtype=np.float64)
                if sub.size == 0:
                    continue
                label = RANKER_LABELS.get(method, method)
                color = RANKER_COLORS.get(method, None)
                interval_row = metric_intervals.loc[
                    metric_intervals["top_pct"].eq(pct) & metric_intervals["method"].eq(method)
                ].iloc[0]

                if np.nanstd(sub) <= 1e-12:
                    ax.axvline(sub[0], color=color, linewidth=2.2, label=label)
                else:
                    ax.hist(sub, bins=32, density=True, alpha=0.35, color=color, label=label)
                    ax.axvline(interval_row["observed_mean"], color=color, linewidth=1.8)
                    ax.axvspan(interval_row["ci_lower"], interval_row["ci_upper"], color=color, alpha=0.10)
            ax.set_title(f"Top {pct}%")
            ax.set_xlabel(metric)
            ax.grid(axis="y", alpha=0.25)
        axes[0].set_ylabel("Bootstrap density")
        axes[-1].legend(loc="best", fontsize=8)
        fig.suptitle(f"Bootstrap distributions of mean {metric}", y=1.03)
        fig.tight_layout()

        metric_slug = metric.replace("/", "_").replace(" ", "_")
        plot_path = BOOTSTRAP_PLOT_DIR / f"bootstrap_mean_{metric_slug}.png"
        fig.savefig(plot_path, bbox_inches="tight")
        pdf.savefig(fig, bbox_inches="tight")
        metric_plot_paths.append(plot_path)

        print(f"Rendered {metric}: {plot_path}")
        display(fig)
        plt.close(fig)

plot_manifest_df = pd.DataFrame({"metric": ENGAGEMENT_METRICS, "plot_path": [str(p) for p in metric_plot_paths]})
display(plot_manifest_df)
print("saved bootstrap plot directory:", BOOTSTRAP_PLOT_DIR)
print("saved bootstrap PDF:", BOOTSTRAP_PDF_OUTPUT_PATH)


In [ ]:
feature_audit_summary = pd.DataFrame()
if len(feature_audit):
    feature_audit_summary = (
        feature_audit
        .groupby(["section", "feature"], as_index=False)
        .agg(count=("count", "sum"), rows=("rows", "sum"))
    )
    feature_audit_summary["share"] = feature_audit_summary["count"] / feature_audit_summary["rows"].clip(lower=1)
    display(feature_audit_summary.sort_values("share", ascending=False).head(30))

group_size_summary = pred_df["rank_group_size"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_dict()

diag = {
    "input_path": str(INPUT_PATH),
    "scored_output_path": str(SCORED_OUTPUT_PATH),
    "singleton_scored_output_path": saved_singleton_scored_rows_path,
    "summary_output_path": str(SUMMARY_OUTPUT_PATH),
    "comparison_output_path": str(COMPARISON_OUTPUT_PATH),
    "summary_wide_output_path": str(SUMMARY_WIDE_OUTPUT_PATH),
    "reporting_table_output_path": str(REPORTING_TABLE_OUTPUT_PATH),
    "reporting_table_top_paths": {str(k): str(v) for k, v in REPORTING_TABLE_TOP_PATHS.items()},
    "bootstrap_draws_output_path": str(BOOTSTRAP_DRAWS_OUTPUT_PATH),
    "bootstrap_intervals_output_path": str(BOOTSTRAP_INTERVALS_OUTPUT_PATH),
    "bootstrap_ci_table_output_path": str(BOOTSTRAP_CI_TABLE_OUTPUT_PATH),
    "bootstrap_ci_wide_output_path": str(BOOTSTRAP_CI_WIDE_OUTPUT_PATH),
    "bootstrap_ci_wide_top_paths": {str(k): str(v) for k, v in BOOTSTRAP_CI_WIDE_TOP_PATHS.items()},
    "bootstrap_plot_dir": str(BOOTSTRAP_PLOT_DIR),
    "bootstrap_pdf_output_path": str(BOOTSTRAP_PDF_OUTPUT_PATH),
    "head_model_path": str(HEAD_MODEL_PATH),
    "eu_model_path": str(EU_MODEL_PATH),
    "rows_full_random": int(raw_n),
    "rows_singleton_random_before_click_filter": int(singleton_n),
    "rows_singleton_click_dropped": int(singleton_click_n),
    "rows_singleton_noclick_random": int(n),
    "singleton_share_before_click_filter": float(singleton_n / raw_n),
    "singleton_click_drop_share_of_singleton": float(singleton_click_n / singleton_n),
    "analysis_share_of_all_random": float(n / raw_n),
    "analysis_share_of_singleton_random": float(n / singleton_n),
    "singleton_filter_col": SINGLETON_FILTER_COL,
    "click_filter_col": CLICK_FILTER_COL,
    "top_pcts": TOP_PCTS,
    "bootstrap_n": int(BOOTSTRAP_N),
    "bootstrap_alpha": float(BOOTSTRAP_ALPHA),
    "bootstrap_random_seed": int(BOOTSTRAP_RANDOM_SEED),
    "write_filtered_scored_rows": bool(WRITE_FILTERED_SCORED_ROWS),
    "write_bootstrap_draws": bool(WRITE_BOOTSTRAP_DRAWS),
    "removed_stale_outputs": removed_stale_outputs,
    "removed_optional_outputs": removed_optional_outputs,
    "bootstrap_note": "Rows are resampled with replacement within each fixed ranker/top-percent selected set to estimate mean outcome intervals.",
    "engagement_metrics": ENGAGEMENT_METRICS,
    "model_score_specs": [
        {"method": method, "score_col": score_col, "rank_col": rank_col}
        for method, score_col, rank_col in MODEL_SCORE_SPECS
    ],
    "head_score_formula": "pred_p_complete + pred_p_long + pred_p_rewatch - pred_p_neg",
    "completion_only_score_formula": "pred_p_complete",
    "eu_score_formula": "pred_EU from EU MLP trained on KuaiRec notebook-07 EU labels",
    "global_quantile_note": "Top p% is computed globally within singleton/no-click KuaiRand random rows by each score.",
    "rank_group_note": "The analysis keeps only original timestamp exposure blocks with one video and drops clicked rows, using is_singleton_user_time == 1 and is_click != 1.",
    "rank_group_size_summary": {k: float(v) for k, v in group_size_summary.items()},
    "num_rank_groups": int(pred_df[rank_group_cols].drop_duplicates().shape[0]) if rank_group_cols else None,
    "num_non_singleton_rank_groups": int((pred_df.groupby(rank_group_cols).size() > 1).sum()) if rank_group_cols else None,
    "feature_audit_top": feature_audit_summary.sort_values("share", ascending=False).head(20).to_dict(orient="records") if len(feature_audit_summary) else [],
}
DIAGNOSTIC_OUTPUT_PATH.write_text(json.dumps(diag, indent=2, default=str) + "\n")

print("saved diagnostics:", DIAGNOSTIC_OUTPUT_PATH)
print(json.dumps({k: diag[k] for k in ["rows_full_random", "rows_singleton_random_before_click_filter", "rows_singleton_click_dropped", "rows_singleton_noclick_random", "analysis_share_of_all_random", "analysis_share_of_singleton_random", "top_pcts", "bootstrap_n", "engagement_metrics", "rank_group_size_summary", "num_non_singleton_rank_groups"]}, indent=2))

## Interpretation Notes

Lower `is_hate` is better; higher values are generally better for the other selected engagement metrics. The output tables compare three rankers: predicted EU, the naive four-head score, and completion-only ranking.

The analysis sample is `is_singleton_user_time == 1` and `is_click != 1`, so it uses only original exposure timestamps where exactly one video was recorded and excludes clicked interactions that are not comparable to the target autoplay UI. In the current scored KuaiRand random sample, the singleton filter keeps **34,608 of 1,149,139 random rows**; dropping **8,459** clicked singleton rows leaves **26,149** analysis rows. Each top percentage is computed globally within this cleaned singleton/no-click sample, not within a session/slate.

Bootstrap intervals use 500 row-bootstrap draws within each fixed top-k selected set. The plots show the bootstrap distribution of the mean metric for the three rankers together, separately for top 5%, 10%, and 20%.